In [0]:
%sql 
select order_id, order_status, order_purchase_timestamp
from demo_catalog.default.orders_2025 limit 1;

order_id,order_status,order_purchase_timestamp
720fd9e9-cdeb-4dec-a187-f71586eb085a,canceled,2025-12-27T07:07:20.000Z


In [0]:
--query 1 OLTP
SELECT * 
FROM demo_catalog.default.orders_2025
WHERE order_id = "720fd9e9-cdeb-4dec-a187-f71586eb085a"; 


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27T07:07:20.000Z,2025-12-27T08:33:20.000Z,null,null,2026-01-04T08:33:20.000Z


In [0]:
--query 2 OLAP
SELECT
date_format(order_purchase_timestamp, 'yyyy-MM') as order_month,
order_status,
count(*) as total_order
from demo_catalog.default.orders_2025
group by order_month, order_status
order by order_month, order_status;

order_month,order_status,total_order
2025-01,canceled,789
2025-01,delivered,15213
2025-02,canceled,851
2025-02,delivered,15751
2025-03,canceled,831
2025-03,delivered,15959
2025-04,canceled,771
2025-04,delivered,15047
2025-05,canceled,1290
2025-05,delivered,26137


In [0]:
--Creating the table copy
CREATE TABLE order_2025_copy as SELECT * from demo_catalog.default.orders_2025;

num_affected_rows,num_inserted_rows


In [0]:
-- checking the table
select * from order_2025_copy where order_id = "720fd9e9-cdeb-4dec-a187-f71586eb085a"

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27T07:07:20.000Z,2025-12-27T08:33:20.000Z,null,null,2026-01-04T08:33:20.000Z


In [0]:
-- Finding the delevered orders limiting to one
select * from order_2025_copy where order_status = 'delivered' limit 1;

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e6441daf-f23c-4a2e-9c72-90403032e1b6,b0f07e35-4fc4-4119-8abf-fc5bc04eaba3,delivered,2025-10-08T10:04:45.000Z,2025-10-08T15:11:45.000Z,2025-10-09T15:11:45.000Z,2025-10-16T15:11:45.000Z,2025-10-18T15:11:45.000Z


In [0]:
-- changing the ordered status of previous delivered 
UPDATE order_2025_copy SET order_status = "canceled" WHERE order_id = "e6441daf-f23c-4a2e-9c72-90403032e1b6"

num_affected_rows
1


In [0]:
-- checking the changed the status
select * from order_2025_copy where order_id = "e6441daf-f23c-4a2e-9c72-90403032e1b6"

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e6441daf-f23c-4a2e-9c72-90403032e1b6,b0f07e35-4fc4-4119-8abf-fc5bc04eaba3,canceled,2025-10-08T10:04:45.000Z,2025-10-08T15:11:45.000Z,2025-10-09T15:11:45.000Z,2025-10-16T15:11:45.000Z,2025-10-18T15:11:45.000Z


In [0]:
-- checking the versioning
DESCRIBE HISTORY order_2025_copy;


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-10-07T01:37:06.000Z,76625678954902,rabinbam9848@gmail.com,UPDATE,"Map(predicate -> [""(order_id#11550 = e6441daf-f23c-4a2e-9c72-90403032e1b6)""])",null,List(2486464644741104),7e67c6fa-6ad9-476b-9bb2-a4827c5e0c9f,1007-010644-eqhadqgf-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5922, numDeletionVectorsUpdated -> 0, scanTimeMs -> 3182, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 3209, rewriteTimeMs -> 2692)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-07T01:32:29.000Z,76625678954902,rabinbam9848@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2486464644741104),0c07749a-7445-4735-98ce-af12d4866d38,1007-010644-eqhadqgf-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 283573, numOutputBytes -> 18780094)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
-- particular version check
select * from order_2025_copy version as of 0 where order_id = "e6441daf-f23c-4a2e-9c72-90403032e1b6";

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e6441daf-f23c-4a2e-9c72-90403032e1b6,b0f07e35-4fc4-4119-8abf-fc5bc04eaba3,delivered,2025-10-08T10:04:45.000Z,2025-10-08T15:11:45.000Z,2025-10-09T15:11:45.000Z,2025-10-16T15:11:45.000Z,2025-10-18T15:11:45.000Z
